In [ ]:
Oracle AI Data Platform v1.0

Copyright © 2025, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# HubSpot Connector Samples

Ingestion samples for HubSpot using Python and the HubSpot REST API through the `aidp-connector-hubspot` package. AIDP has no built-in HubSpot connector type. Replace all placeholders before running a sample.

## Prerequisites

1. A HubSpot service key or private app token with the scopes `crm.objects.contacts.read`, `crm.objects.companies.read` and `crm.objects.deals.read`.
2. A **Secret Token** credential in the Credential Store that holds the token under the key `<KEY>`.
3. Outbound HTTPS access from the cluster to `api.hubapi.com`.
4. An existing target catalog. The schema is created if it is missing.
5. The `aidp-connector-hubspot` wheel on the cluster, see the next section.

## Install the Package

Get the source from [oracle-aidp-connectors](https://github.com/arbisoft/oracle-aidp-connectors). On your machine, run `uv build --wheel` in the folder `src/aidp_connector_hubspot` to build the wheel. Upload the wheel to your workspace. Then open the cluster **Library** tab, choose **Install Library** and **Workspace**, select the wheel and restart the cluster. The package installs `requests` and `pyyaml` with it.

To install for this notebook only, uncomment the `%pip` line below and set the path.

In [ ]:
# %pip install /Workspace/<path-to>/aidp_connector_hubspot-<version>-py3-none-any.whl

In [ ]:
import aidp_connector_hubspot

print(aidp_connector_hubspot.__version__)

## Configuration

Save this file in your workspace as `hubspot_ingest.yaml`. It holds no secret. The target names may contain only letters, digits and underscores.

```yaml
hubspot:
  credential_name: <CREDENTIAL_NAME>
  credential_key: <KEY>
target:
  catalog: <CATALOG>
  schema: <SCHEMA>
sync:
  mode: incremental
  objects: [contacts, companies, deals, associations]
```

In [ ]:
from aidp_connector_hubspot import format_summary, load_config, raise_on_failure, read_token, run

config = load_config("/Workspace/<path>/hubspot_ingest.yaml")

## Ingestion Sample

Read the token from the Credential Store and run the sync. Each object is synced on its own, so one failure does not stop the others. The last call raises an error if any object failed.

In [ ]:
token = read_token(config.hubspot, aidputils.secrets.get)

summaries = run(spark, config, token)
print(format_summary(summaries))
raise_on_failure(summaries)

## Incremental and Full Refresh

The default mode is `incremental`. It reads only the records that changed since the last successful run. An object with no watermark is loaded in full. Set `sync.mode` to `full` in the config, or pass `mode_override` to `run`, to reload every record and overwrite the tables for one run.

In [ ]:
summaries = run(spark, config, token, mode_override="full")
print(format_summary(summaries))

## Query the Tables

The tables are written to `<CATALOG>.<SCHEMA>`. If you set `target.table_prefix`, add it before the table name.

In [ ]:
spark.table("<CATALOG>.<SCHEMA>.contacts").show()
spark.table("<CATALOG>.<SCHEMA>.deals").show()

## Connector Options

| Parameter name | Valid values | Mandatory | Description |
| --- | --- | --- | --- |
| `hubspot.credential_name` | Credential Store entry name | N | Entry that holds the HubSpot token. If not set, the token is read from the environment variable named by `hubspot.token_env`. |
| `hubspot.credential_key` | Key name in the credential | N | Key that holds the token inside the credential. Default is `secret`. |
| `hubspot.token_env` | Environment variable name | N | Environment variable used when no credential is named. Default is `HUBSPOT_TOKEN`. |
| `hubspot.api_version` | Dated HubSpot CRM API version | N | Version used in every request path. Default is `2026-03`. |
| `hubspot.requests_per_second` | Number greater than 0 | N | Client-side request limit. Default is `4`. HubSpot search allows 5 per second. |
| `target.catalog` | Existing catalog name | Y | Target catalog. Letters, digits and underscores only. |
| `target.schema` | Schema name | Y | Target schema. Created if it is missing. Letters, digits and underscores only. |
| `target.table_prefix` | Letters, digits and underscores | N | Prefix added to every table name. Default is empty. |
| `sync.mode` | `incremental`, `full` | N | `incremental` reads only changes since the last run. `full` reloads and overwrites. Default is `incremental`. |
| `sync.objects` | `contacts`, `companies`, `deals`, `associations` | Y | Objects to ingest. Associations are read from the ids in the contacts and deals tables. |
| `sync.overlap_seconds` | Number of seconds, 0 or more | N | How far before the watermark an incremental run reads again. Default is `300`. |